In [1]:
from func import *
from models import *

In [2]:
train_df = pd.read_csv("new_train_df_clean.csv")
test_df = pd.read_csv("new_test_df_clean.csv")

In [3]:
# select features
zip_dummy_cols = [col for col in test_df.columns if col.startswith('ZIP_prefix_')]
selected_features = [
    'Flooring_target_mean',
    'ViewYN',
    'PoolPrivateYN',
    'LivingArea_std',
    'AttachedGarageYN',
    'ParkingTotal',
    'Age',
    'BathroomsTotalInteger',
    'BedroomsTotal',
    'FireplaceYN',
    'Levels_final_One',
    'Levels_final_Two',
    'Levels_final_ThreeOrMore',
    'MainLevelBedrooms',
    'NewConstructionYN',
    'GarageSpaces',
    'HighSchoolDistrict_target_mean',
    'LotSizeSquareFeet_std',
    'AssociationFeeFrequency_Monthly',
    'AssociationFeeFrequency_Quarterly',
    'AssociationFeeFrequency_SemiAnnually',
    'Stories_2.0',
    'AssociationFee_std',
    'dist_to_coast_km',
    'Latitude',
    'Longitude'
] + zip_dummy_cols


In [9]:
X_train = train_df[selected_features]
y_train = train_df["log_ClosePrice"]

X_test = test_df[selected_features]
y_test = test_df["log_ClosePrice"]

# RF

In [10]:
# Train final model with best params
rf_best_params = {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': None, 'max_depth': None, 'bootstrap': True}
best_rf = RandomForestRegressor(**rf_best_params, random_state=222, n_jobs=-1)
best_rf.fit(X_train, y_train)

# Predict on train and test
y_train_pred_rf_log_best = best_rf.predict(X_train)
y_pred_rf_log_best = best_rf.predict(X_test)

# XGB

In [11]:
# Train final model with best params
xgb_best_params = {'subsample': 0.8, 'reg_lambda': 5.0, 'reg_alpha': 0, 'n_estimators': 800, 'min_child_weight': 1, 'max_depth': 9, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 0.7}
best_xgb = XGBRegressor(**xgb_best_params, random_state=222, n_jobs=-1)
best_xgb.fit(X_train, y_train)

# Predict on train and test
y_train_pred_xgb_log_best = best_xgb.predict(X_train)
y_pred_xgb_log_best = best_xgb.predict(X_test)

# Light gbm

In [12]:
# Train final model
lgbm_best_params = {'subsample': 0.6, 'reg_lambda': 0.01, 'reg_alpha': 0.1, 'num_leaves': 80, 'n_estimators': 1000, 'min_child_samples': 40, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

best_lgbm = LGBMRegressor(**lgbm_best_params, random_state=222, n_jobs=-1, verbosity=-1)
best_lgbm.fit(X_train, y_train)

# Predict on train and test
y_train_pred_lgbm_log_best = best_lgbm.predict(X_train)
y_pred_lgbm_log_best = best_lgbm.predict(X_test)

# Stack and weighted

In [13]:
# Simple equal-weight average
y_train_pred_log_stack = (y_train_pred_lgbm_log_best + y_train_pred_xgb_log_best + y_train_pred_rf_log_best) / 3
y_pred_log_stack = (y_pred_lgbm_log_best + y_pred_xgb_log_best + y_pred_rf_log_best) / 3

# Weighted average
weights = [0.8, 0.1, 0.1]  # e.g., LightGBM highest weight
y_train_pred_log_stack_weighted = (
    weights[0] * y_train_pred_lgbm_log_best +
    weights[1] * y_train_pred_xgb_log_best +
    weights[2] * y_train_pred_rf_log_best
)
y_pred_log_stack_weighted = (
    weights[0] * y_pred_lgbm_log_best +
    weights[1] * y_pred_xgb_log_best +
    weights[2] * y_pred_rf_log_best
)


stack_metric = compute_metrics(y_test, y_pred_log_stack, y_train, y_train_pred_log_stack)
weighted_stack_metric = compute_metrics(y_test, y_pred_log_stack_weighted, y_train, y_train_pred_log_stack_weighted)

# Stack (StackingRegressor)

In [17]:
from sklearn.ensemble import StackingRegressor


# ---------- Stack them ----------
stack = StackingRegressor(
    estimators=[
        ("lgbm", LGBMRegressor(**lgbm_best_params, random_state=222, n_jobs=-1, verbosity=-1)),
        ("xgb", XGBRegressor(**xgb_best_params, random_state=222, n_jobs=-1))
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

stack.fit(X_train, y_train)

# Predict log target
y_train_pred = stack.predict(X_train)
y_pred = stack.predict(X_test)

stack_reg_metrics = compute_metrics(y_test, y_pred, y_train, y_train_pred)

# Residual model (StackingRegressor)

In [20]:
residual_model = XGBRegressor(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
)

residual = y_train - y_train_pred

residual_model.fit(X_train, residual)

pred1 = stack.predict(X_test)
pred2 = residual_model.predict(X_test)

train_pred1 = stack.predict(X_train)
train_pred2 = residual_model.predict(X_train)

y_residual_pred = pred1 + pred2
y_train_residual_pred = train_pred1 + train_pred2

residual_metrics = compute_metrics(y_test, y_residual_pred, y_train, y_train_residual_pred)

# Residual model (weighted)

In [23]:
weighted_residual_model = XGBRegressor(
    max_depth=3,
    learning_rate=0.03,
    n_estimators=400,
    subsample=0.8,
    colsample_bytree=0.8,
)

weighted_residual = y_train - y_train_pred_log_stack_weighted

weighted_residual_model.fit(X_train, weighted_residual)

weighted_pred2 = residual_model.predict(X_test)

weighted_train_pred2 = weighted_residual_model.predict(X_train)

weighted_y_residual_pred = y_pred_log_stack_weighted + weighted_pred2
weighted_y_train_residual_pred = y_train_pred_log_stack_weighted + weighted_train_pred2

weighted_residual_metrics = compute_metrics(y_test, weighted_y_residual_pred, y_train, weighted_y_train_residual_pred)

# Summary

In [24]:
models_summary = {
    "Stack": stack_metric,
    "Weighted Stack": weighted_stack_metric,
    "Stack regressor": stack_reg_metrics,
    "Residual Model": residual_metrics,
    "Weighted Residual Model": weighted_residual_metrics,
}

df = pd.DataFrame.from_dict(models_summary).transpose()
df

,R2(log),R2,MAPE,MdAPE,RMSE,MAE,Bias(mean residual),APE_95pct,APE_99pct,APE_max,Train_R2(log),Test_R2(log),R2_gap
Stack,0.932568,0.890669,11.230285,7.457933,305818.374417,140326.549993,-5056.643937,34.033539,61.741349,185.555936,0.969052,0.932568,0.036484
Weighted Stack,0.934973,0.896294,11.105935,7.409131,297846.352376,138097.671545,-360.553249,34.072596,59.999996,183.664249,0.969916,0.934973,0.034943
Stack regressor,0.935153,0.896472,11.121016,7.450513,297591.563569,138680.329466,1314.854390,34.106736,59.989659,178.392489,0.968359,0.935153,0.033206
Residual Model,0.935274,0.896314,11.108007,7.429436,297817.729724,138577.109259,604.192538,34.240702,59.686378,177.923901,0.968552,0.935274,0.033278
Weighted Residual Model,0.935059,0.896057,11.099139,7.409885,298186.410041,138083.178199,-1061.178379,34.037327,59.499555,183.623407,0.970098,0.935059,0.035039
